In [0]:
from pyspark.sql.functions import col, explode, to_date, year, when, lit, avg, count, desc, split, trim, substring, expr
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql.functions import sum as _sum

file = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"
df_steam = spark.read.json(file)
df_steam.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df_steam.show(5)

+--------------------+-------+
|                data|     id|
+--------------------+-------+
|{10, [Multi-playe...|     10|
|{1000000, [Single...|1000000|
|{1000010, [Single...|1000010|
|{1000030, [Multi-...|1000030|
|{1000040, [Single...|1000040|
+--------------------+-------+
only showing top 5 rows


In [0]:
df_clean = df_steam.select(
    col("id").alias("appid"),
    col("data.name").alias("name"),
    col("data.publisher").alias("publisher"),
    col("data.price").cast("float").alias("price"),
    col("data.positive").cast("int").alias("positive_ratings"),
    col("data.negative").cast("int").alias("negative_ratings"),
    col("data.release_date").alias("release_date_raw"),
    col("data.genre").alias("genres"),
    col("data.languages").alias("languages_raw"),
    col("data.platforms").alias("platforms"),
    col("data.required_age").alias("required_age")
)

# Extraction de l'année avec gestion des erreurs avec expr("try_cast(...)") pour ignorer les valeurs malformées
df_clean = df_clean.withColumn(
    "release_year", 
    expr("try_cast(substring(release_date_raw, 1, 4) as int)")
)

# Calcul du score ratio
df_clean = df_clean.withColumn("total_reviews", col("positive_ratings") + col("negative_ratings")) \
                   .withColumn("score_ratio", 
                               when(col("total_reviews") == 0, 0)
                               .otherwise((col("positive_ratings") / col("total_reviews")) * 100))

# Filtrage des lignes où l'année est nulle ou l'éditeur est vide
df_clean = df_clean.filter(col("release_year").isNotNull())

display(df_clean.select("name", "publisher", "release_year").limit(10))

name,publisher,release_year
Counter-Strike,Valve,2000
ASCENXION,PsychoFlux Entertainment,2021
Crown Trick,"Team17, NEXT Studios",2020
"Cook, Serve, Delicious! 3?!",Vertigo Gaming Inc.,2020
细胞战争,DoubleC Games,2019
Zengeon,2P Games,2019
干支セトラ 陽ノ卷｜干支etc. 陽之卷,Starship Studio,2019
Jumping Master(跳跳大咖),重庆环游者网络科技,2019
Cube Defender,Simon Codrington,2019
Tower of Origin2-Worm's Nest,Villain Role,2021


In [0]:
# Qui a publié le plus de jeux ?
df_top_publishers = df_clean.groupBy("publisher") \
    .count() \
    .orderBy(col("count").desc())

display(df_top_publishers.limit(15))

publisher,count
Big Fish Games,422
8floor,202
SEGA,162
Strategy First,151
Square Enix,140
Choice of Games,140
HH-Games,132
,132
Sekai Project,132
Ubisoft,126


Databricks visualization. Run in Databricks to view.

In [0]:
# Agrégation par année
df_years = df_clean.groupBy("release_year") \
    .count() \
    .orderBy("release_year")

display(df_years)

release_year,count
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


Databricks visualization. Run in Databricks to view.

In [0]:
# Valeurs étranges sur les cat prix, donc check si les prix sont en centimes peut-être? Check sur Cyberpunk ou AC
df_clean.filter(col("name").like("%Cyberpunk%") | col("name").like("%Assassin's Creed%")) \
        .select("name", "price") \
        .show(10)

+-------------------------+------+
|                     name| price|
+-------------------------+------+
|           Cyberpunk 2077|5999.0|
|Sense - 不祥的预感: A ...|1999.0|
|            Cyberpunk SFX|1199.0|
|     CrashMetal - Cybe...| 499.0|
|     Assassin's Creed:...|1999.0|
|        Cyberpunk Madness|  99.0|
|     Twilight Town: A ...| 299.0|
|       Cyberpunk Fighting|  99.0|
|          Cyberpunk Girls| 699.0|
|     Assassin's Creed ...|1999.0|
+-------------------------+------+
only showing top 10 rows


In [0]:
# Prix en centimes
df_prices = df_clean.withColumn("price_euro", col("price") / 100)

# Création catégories
df_prices = df_prices.withColumn("price_category", 
    when(col("price_euro") == 0, "0 - Gratuit")
    .when((col("price_euro") > 0) & (col("price_euro") <= 10), "1 - Petit budget (<10€)")
    .when((col("price_euro") > 10) & (col("price_euro") <= 30), "2 - Standard (10-30€)")
    .when((col("price_euro") > 30) & (col("price_euro") <= 60), "3 - AAA (30-60€)")
    .otherwise("4 - Premium (>60€)"))

df_price_distrib = df_prices.groupBy("price_category").count().orderBy("price_category")

display(df_price_distrib)

price_category,count
0 - Gratuit,7711
1 - Petit budget (<10€),35925
2 - Standard (10-30€),10793
3 - AAA (30-60€),1041
4 - Premium (>60€),122


Databricks visualization. Run in Databricks to view.

In [0]:
# Langues
df_langs = df_clean.withColumn("lang_list", split(col("languages_raw"), ",")) \
                   .withColumn("language", explode(col("lang_list"))) \
                   .withColumn("language", trim(col("language")))

df_top_langs = df_langs.groupBy("language").count().orderBy(desc("count"))

display(df_top_langs.limit(15))

language,count
English,55017
German,13981
French,13391
Russian,12901
Simplified Chinese,12771
Spanish - Spain,12200
Japanese,10354
Italian,9274
Portuguese - Brazil,6738
Korean,6593


Databricks visualization. Run in Databricks to view.

In [0]:
# Extraction des âges(ex: "7+"" devient "7")
df_age_clean = df_clean.withColumn(
    "age_digits", 
    regexp_extract(col("required_age"), r"(\d+)", 1).cast("int")
)

# Création labels PEGI
df_age_labeled = df_age_clean.withColumn("pegi_category", 
    when(col("age_digits") == 0, "Tout public")
    .when((col("age_digits") > 0) & (col("age_digits") <= 3), "0-3 ans")
    .when((col("age_digits") > 3) & (col("age_digits") <= 7), "3-7 ans")
    .when((col("age_digits") > 7) & (col("age_digits") <= 12), "7-12 ans")
    .when((col("age_digits") > 12) & (col("age_digits") <= 16), "12-16 ans")
    .when((col("age_digits") > 16) & (col("age_digits") <= 30), "16-30 ans")
    .otherwise("Non classé") # Pour les valeurs abhérantes
)

df_age_final = df_age_labeled.groupBy("pegi_category").count().orderBy("pegi_category")

display(df_age_final)

pegi_category,count
0-3 ans,3
12-16 ans,336
16-30 ans,260
3-7 ans,8
7-12 ans,43
Non classé,5
Tout public,54937


Databricks visualization. Run in Databricks to view.

In [0]:
# Transformation colonne "genres" (str) en array
df_clean_with_array = df_clean.withColumn(
    "genres_array", 
    split(regexp_replace(col("genres"), r'[\[\]\"]', ""), ",")
)

# "explode"
df_genres_exploded = df_clean_with_array.withColumn("genre_single", explode(col("genres_array")))

# Nettoyage espaces vides
df_genres_exploded = df_genres_exploded.withColumn("genre_single", trim(col("genre_single")))

# Genres les plus représentés
df_genre_rep = df_genres_exploded.groupBy("genre_single").count().orderBy(desc("count"))
display(df_genre_rep)

genre_single,count
Indie,39660
Action,23729
Casual,22066
Adventure,21402
Strategy,10878
Simulation,10824
RPG,9527
Early Access,6142
Free to Play,3390
Sports,2662


In [0]:
# Ratio de satisfaction par genre
df_genre_score = df_genres_exploded.groupBy("genre_single") \
    .agg(
        avg("score_ratio").alias("avg_satisfaction"), 
        count("appid").alias("nb_games")
    ) \
    .filter(col("nb_games") > 50) \
    .orderBy(desc("avg_satisfaction"))

display(df_genre_score)

genre_single,avg_satisfaction,nb_games
Casual,74.01205705158563,22066
Indie,73.98339924741973,39660
Game Development,73.77875884249147,159
Adventure,73.75012845024904,21402
RPG,72.95419369701482,9527
Action,72.80231749178161,23729
Free to Play,72.08730675539555,3390
Web Publishing,71.96752177479995,89
Design & Illustration,71.85601895141501,405
Strategy,71.8213075709716,10878


In [0]:
# Jeux sur quelles plateformes?
df_platforms = df_clean.select(
    _sum(col("platforms.windows").cast("int")).alias("Windows"),
    _sum(col("platforms.mac").cast("int")).alias("Mac"),
    _sum(col("platforms.linux").cast("int")).alias("Linux")
)

df_platforms_pie = df_platforms.select(
    expr("stack(3, 'Windows', Windows, 'Mac', Mac, 'Linux', Linux) as (Platform, Game_Count)")
)

display(df_platforms_pie)

Platform,Game_Count
Windows,55577
Mac,12742
Linux,8448


Databricks visualization. Run in Databricks to view.

In [0]:
# Est-ce que les jeux plus chers sont mieux notés ?
df_price_quality = df_prices.groupBy("price_category") \
    .agg(
        avg("score_ratio").alias("note_moyenne"),
        count("appid").alias("nombre_de_jeux")
    ) \
    .orderBy("price_category")

display(df_price_quality)

price_category,note_moyenne,nombre_de_jeux
0 - Gratuit,69.52356390798138,7711
1 - Petit budget (<10€),73.21494591645374,35925
2 - Standard (10-30€),76.6612469648693,10793
3 - AAA (30-60€),76.9325471804733,1041
4 - Premium (>60€),71.86872016234646,122


Databricks visualization. Run in Databricks to view.